# Notebook that matches the gt with lables

So for each split we have a csv with bounding box coordinates as well as the correct species of the animal

## Required Functions

In [8]:
import pandas as pd
from utils import PROJECT_ROOT, SPLITS, IMG_SIZES, resolve_frame_paths

In [9]:
def yolo_to_xyxy(x_center: float, y_center: float, width: float, height: float, image_size:tuple[int, int]=IMG_SIZES) -> tuple[float, float, float, float]:
    """Takes yolo format and converts it actual pixel coorindates on an image

    Args:
        x_center (float): yolo x
        y_center (float): yolo y
        width (float): with of yolo box
        height (float): hight of yolo box
        image_width (int): width of actual image in px
        image_height (int): height of actual image in px

    Returns:
        tuple[float, float, float, float]: actuall coordinates of bounding box in px (bottom left corner)
    """
    x_center_px = x_center * image_size[0]
    y_center_px = y_center * image_size[1]
    width_px = width * image_size[0]
    height_px = height * image_size[1]

    x_min = x_center_px - (width_px / 2)
    y_min = y_center_px - (height_px / 2)
    return x_min, y_min, width_px, height_px

In [10]:
def convert_labels_matched_to_df(frame_name:str, image_size:tuple[int, int]=IMG_SIZES,)->pd.DataFrame:
    _, _, label_path = resolve_frame_paths(frame_name)

    txt_records = []
    with open(label_path, "r", encoding="utf-8") as f:
        for txt_idx, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            _, x_c, y_c, w, h = line.split()
            x_c = float(x_c)
            y_c = float(y_c)
            w = float(w)
            h = float(h)

            x_min, y_min, width_px, heigth_px = yolo_to_xyxy(x_c, y_c, w, h, image_size)
            x2 = x_min + width_px
            y2 =y_min + heigth_px

            txt_records.append(
                {
                    "txt_idx": txt_idx,
                    "txt_x1": int(x_min),
                    "txt_y1": int(y_min),
                    "txt_x2": int(x2),
                    "txt_y2": int(y2),
                    "txt_xc": int(x_c),
                    "txt_yc": int(y_c),
                    "txt_h":  int(heigth_px), 
                    "txt_w":  int(width_px)
                }
            )

    txt_boxes_df = pd.DataFrame(txt_records)
    return txt_boxes_df

convert_labels_matched_to_df("104_1282")

,txt_idx,txt_x1,txt_y1,txt_x2,txt_y2,txt_xc,txt_yc,txt_h,txt_w
0,0,731,683,777,724,0,0,41,45
1,1,598,657,646,703,0,0,45,48


In [11]:
def match_gt_to_txt_boxes(
    gt_df: pd.DataFrame,
    txt_boxes_df:pd.DataFrame,
    min_iou=0.0,
):
    """Match GT pixel boxes to YOLO txt boxes for one frame does not load images.

    Args:
        gt_df: DataFrame containing GT rows with columns frame, bb_left, bb_top, bb_width, bb_height. filtered for the correct frames
        frame_name: the name of the frame, consisting of flight_frameID e.g. 102_3444 
        image_size: (width, height) used to convert YOLO normalized coords to pixels.
        min_iou: Minimum IoU required to keep a match.

    Returns:
        pandas.Dataframe: a dataframe with the columns: framename, class, x_min, y_min, width_px, height_px
    """

    gt_rows = gt_df.reset_index().rename(columns={"index": "gt_row_idx"})
    gt_rows["gt_x1"] = gt_rows["bb_left"].astype(float)
    gt_rows["gt_y1"] = gt_rows["bb_top"].astype(float)
    gt_rows["gt_x2"] = gt_rows["gt_x1"] + gt_rows["bb_width"].astype(float)
    gt_rows["gt_y2"] = gt_rows["gt_y1"] + gt_rows["bb_height"].astype(float)
    gt_rows["gt_xc"] = (gt_rows["gt_x1"] + gt_rows["gt_x2"]) / 2.0
    gt_rows["gt_yc"] = (gt_rows["gt_y1"] + gt_rows["gt_y2"]) / 2.0

    def _iou(a_x1, a_y1, a_x2, a_y2, b_x1, b_y1, b_x2, b_y2):
        inter_x1 = max(a_x1, b_x1)
        inter_y1 = max(a_y1, b_y1)
        inter_x2 = min(a_x2, b_x2)
        inter_y2 = min(a_y2, b_y2)

        inter_w = max(0.0, inter_x2 - inter_x1)
        inter_h = max(0.0, inter_y2 - inter_y1)
        inter_area = inter_w * inter_h

        area_a = max(0.0, a_x2 - a_x1) * max(0.0, a_y2 - a_y1)
        area_b = max(0.0, b_x2 - b_x1) * max(0.0, b_y2 - b_y1)
        union = area_a + area_b - inter_area
        return 0.0 if union <= 0 else inter_area / union

    candidates = []
    if not txt_boxes_df.empty:
        for _, gt_row in gt_rows.iterrows():
            for _, txt_row in txt_boxes_df.iterrows():
                iou = _iou(
                    gt_row["gt_x1"], gt_row["gt_y1"], gt_row["gt_x2"],gt_row["gt_y2"],
                    txt_row["txt_x1"], txt_row["txt_y1"], txt_row["txt_x2"], txt_row["txt_y2"],
                )
                if iou < min_iou:
                    continue

                center_dist = ((gt_row["gt_xc"] - txt_row["txt_xc"]) ** 2 + (gt_row["gt_yc"] - txt_row["txt_yc"]) ** 2) ** 0.5
                candidates.append(
                    {
                        "gt_row_idx": int(gt_row["gt_row_idx"]),
                        "txt_idx": int(txt_row["txt_idx"]),
                        "iou": float(iou),
                        "center_dist": float(center_dist),
                    }
                )

    candidates_df = pd.DataFrame(candidates)

    assigned_gt = set()
    assigned_txt = set()
    chosen = []

    if not candidates_df.empty:
        candidates_df = candidates_df.sort_values(["iou", "center_dist"], ascending=[False, True]).reset_index(drop=True)
        for _, cand in candidates_df.iterrows():
            g = int(cand["gt_row_idx"])
            t = int(cand["txt_idx"])
            if g in assigned_gt or t in assigned_txt:
                continue
            assigned_gt.add(g)
            assigned_txt.add(t)
            chosen.append(cand.to_dict())

    chosen_df = pd.DataFrame(chosen)

    matches_df = gt_rows[["gt_row_idx", "class_id", "species"]].copy()
    if chosen_df.empty:
        matches_df["txt_idx"] = pd.NA
        matches_df["txt_class_id"] = pd.NA
        matches_df["iou"] = 0.0
        matches_df["center_dist"] = pd.NA
    else:
        matches_df = matches_df.merge(chosen_df, on="gt_row_idx", how="left")
        matches_df = matches_df.merge(txt_boxes_df, on="txt_idx", how="right")
        matches_df["iou"] = matches_df["iou"].fillna(0.0)

    matches_df.sort_values(["iou", "center_dist"], ascending=[False, True], na_position="last", inplace=True)
    return matches_df

## Reading the files and merging them

In [12]:
headings = []
with open("../data/gt_headings.txt", "r") as f:
    headings = f.readline().split(";")

In [ ]:
# For testing
# frame = "17_26681"
# flight_id, frame_id = frame.split("_")

# gt = pd.read_csv(PROJECT_ROOT/f"data/gt/{flight_id}_gt.txt", names=headings)
# yolo = convert_labels_matched_to_df(frame)
# result = match_gt_to_txt_boxes(gt[gt["frame"] == int(frame_id)], yolo)
# # result.drop(["gt_row_idx", "txt_idx", "iou", "center_dist", "txt_xc", "txt_yc"], axis=1, inplace=True)
# result.insert(0, "flight", flight_id)
# result.insert(1, "frame", frame_id)
# result

,flight,frame,gt_row_idx,class_id,species,txt_idx,iou,center_dist,txt_x1,txt_y1,txt_x2,txt_y2,txt_xc,txt_yc,txt_h,txt_w
1,17,26681,18568.0,11.0,Cervus elaphus (Red deer),1.0,0.468841,898.965656,844,234,892,286,0,0,51,47
0,17,26681,NaN,NaN,NaN,0.0,0.000000,NaN,867,357,916,401,0,0,44,49
2,17,26681,NaN,NaN,NaN,2.0,0.000000,NaN,802,390,853,435,0,0,44,50
3,17,26681,NaN,NaN,NaN,3.0,0.000000,NaN,893,399,946,446,0,0,46,53
4,17,26681,NaN,NaN,NaN,4.0,0.000000,NaN,732,882,779,923,0,0,40,47


In [14]:
# Get all the different frames and their flights with split
files_in_splits = []
for split in SPLITS:
    gt_path = PROJECT_ROOT/f"data/labels_matched_rgb/{split}"
    for f in gt_path.iterdir():
        file = f.name[:-4]
        files_in_splits.append(
            {
                "flight": int(file.split("_")[0]),
                "frame": int(file.split("_")[1]),
                "split": split
            }
        )
all_files = pd.DataFrame(files_in_splits)

In [15]:
# Get a dictionary of the ALL matched animals from all flights
# Dict is split by splits and holsd a df per split
# DF contains the flight, frame, species and bounding box
results_per_split: dict[str, pd.DataFrame] = {}
for split in SPLITS:
    results = []
    files = all_files[all_files["split"] == split]
    for flight_id in files["flight"].unique():
        frames = files[files["flight"] == flight_id]
        gt = pd.read_csv(PROJECT_ROOT/f"data/gt/{flight_id}_gt.txt", names=headings)
        print(f"Processing flight: {flight_id}...(*￣０￣)ノ")
        for frame_id in frames["frame"]:
            yolo = convert_labels_matched_to_df(f"{flight_id}_{frame_id}")
            result = match_gt_to_txt_boxes(gt[gt["frame"] == frame_id], yolo)
            result.drop(["gt_row_idx", "txt_idx", "iou", "center_dist", "txt_xc", "txt_yc"], axis=1, inplace=True)
            result.insert(0, "flight", flight_id)
            result.insert(1, "frame", frame_id)
            results.append(result)
    results_per_split[split] = pd.concat(results, axis=0, ignore_index=True)
# results_per_split

Processing flight: 100...(*￣０￣)ノ
Processing flight: 101...(*￣０￣)ノ
Processing flight: 102...(*￣０￣)ノ
Processing flight: 103...(*￣０￣)ノ
Processing flight: 104...(*￣０￣)ノ
Processing flight: 105...(*￣０￣)ノ
Processing flight: 106...(*￣０￣)ノ
Processing flight: 119...(*￣０￣)ノ
Processing flight: 122...(*￣０￣)ノ
Processing flight: 123...(*￣０￣)ノ
Processing flight: 129...(*￣０￣)ノ
Processing flight: 130...(*￣０￣)ノ
Processing flight: 131...(*￣０￣)ノ
Processing flight: 132...(*￣０￣)ノ
Processing flight: 134...(*￣０￣)ノ
Processing flight: 135...(*￣０￣)ノ
Processing flight: 136...(*￣０￣)ノ
Processing flight: 137...(*￣０￣)ノ
Processing flight: 138...(*￣０￣)ノ
Processing flight: 139...(*￣０￣)ノ
Processing flight: 140...(*￣０￣)ノ
Processing flight: 141...(*￣０￣)ノ
Processing flight: 142...(*￣０￣)ノ
Processing flight: 143...(*￣０￣)ノ
Processing flight: 155...(*￣０￣)ノ
Processing flight: 156...(*￣０￣)ノ
Processing flight: 17...(*￣０￣)ノ
Processing flight: 188...(*￣０￣)ノ
Processing flight: 189...(*￣０￣)ノ
Processing flight: 18...(*￣０￣)ノ
Processing f

In [16]:
results_per_split["train"].sort_values(["flight", "frame"]).reset_index(drop=True)

,flight,frame,class_id,species,txt_x1,txt_y1,txt_x2,txt_y2,txt_h,txt_w
0,1,11726,1.0,Cervus elaphus (Red deer),709,578,749,632,53,40
1,17,14339,5.0,Cervus elaphus (Red deer),456,102,510,146,43,54
2,17,14342,5.0,Cervus elaphus (Red deer),454,115,507,161,46,52
3,17,14343,5.0,Cervus elaphus (Red deer),449,141,506,191,50,56
4,17,14349,5.0,Cervus elaphus (Red deer),447,178,504,228,49,57
...,...,...,...,...,...,...,...,...,...,...
11380,252,6725,15.0,Cervus elaphus (Red deer),721,218,761,260,42,39
11381,252,6725,10.0,Cervus elaphus (Red deer),655,310,699,349,38,43
11382,252,6725,10.0,Cervus elaphus (Red deer),623,224,662,264,40,38
11383,252,6785,10.0,Cervus elaphus (Red deer),658,158,704,198,40,45


In [17]:
for split in SPLITS:
    to_export = results_per_split[split].sort_values(["flight", "frame"])
    to_export[["class_id", "species"]]= to_export.groupby(["flight", "frame"])[["class_id", "species"]].ffill().bfill()
    to_export["class_id"] = to_export["class_id"].astype(int)
    to_export.to_csv(PROJECT_ROOT/f"data/bounding_boxes_with_labels/{split}.csv", index=False)